# Notebook 07 — CNN Training
Trains a 1D-CNN classifier on CIC-DDoS2019 with hyperparameter tuning.

In [ ]:
import subprocess, os
from pathlib import Path

REPO_URL  = 'https://github.com/calvinkatoroy/tugas-akhir-ai-kel-08.git'
REPO_NAME = 'tugas-akhir-ai-kel-08'

cwd = Path.cwd()

if (cwd / '../src').resolve().exists():
    result = subprocess.run(['git', '-C', str((cwd / '..').resolve()), 'pull'],
                            capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')

elif (cwd / 'src').exists():
    result = subprocess.run(['git', 'pull'], capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')
    os.chdir('notebooks')

else:
    repo_path = cwd / REPO_NAME
    if not repo_path.exists():
        print(f'Cloning {REPO_URL} ...')
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    else:
        print('Repo found, pulling latest...')
        subprocess.run(['git', '-C', str(repo_path), 'pull'], capture_output=True)
    os.chdir(repo_path / 'notebooks')

print(f'Working dir: {Path.cwd()}')

In [ ]:
import sys
sys.path.insert(0, '..')

import random
import copy
import numpy as np
import torch
import yaml
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/tugas-akhir-ai")
    SPLITS_DIR = DRIVE_ROOT / "splits"
else:
    import yaml
    with open("../config.yaml") as f:
        _cfg = yaml.safe_load(f)
    DRIVE_ROOT = Path(_cfg["data"]["drive_root"])
    SPLITS_DIR = Path(_cfg["data"]["splits_drive"])

print(f"DRIVE_ROOT: {DRIVE_ROOT}")
print(f"SPLITS_DIR: {SPLITS_DIR}")

In [ ]:
with open("../config.yaml") as f:
    cfg = yaml.safe_load(f)

# CNN uses same seq splits as LSTM/GRU
X_train = np.load(SPLITS_DIR / "X_train_seq.npy")
y_train = np.load(SPLITS_DIR / "y_train_seq.npy")
X_val   = np.load(SPLITS_DIR / "X_val_seq.npy")
y_val   = np.load(SPLITS_DIR / "y_val_seq.npy")
X_test  = np.load(SPLITS_DIR / "X_test_seq.npy")
y_test  = np.load(SPLITS_DIR / "y_test_seq.npy")

n_features = X_train.shape[2]
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
print(f"n_features={n_features}")

## Baseline run (config defaults)

In [ ]:
from src.models.cnn_model import build_cnn
from src.train import train_model
from src.utils import log_experiment

model = build_cnn(cfg, n_features)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

In [ ]:
history_baseline, ckpt_baseline = train_model(
    model, X_train, y_train, X_val, y_val,
    cfg=cfg, model_key='cnn',
    checkpoint_dir='../results/checkpoints',
)

log_experiment({
    'exp_id': 'cnn_01_baseline',
    'model': 'CNN',
    'num_filters': cfg['cnn']['num_filters'],
    'kernel_size': cfg['cnn']['kernel_size'],
    'dropout': cfg['cnn']['dropout'],
    'best_val_f1': round(max(history_baseline['val_f1']), 4),
    'notes': 'baseline — config defaults',
}, csv_path=str(DRIVE_ROOT / "metrics_summary.csv"))

## Hyperparameter Experiments

In [ ]:
def run_cnn_experiment(exp_id, overrides, notes=''):
    exp_cfg = copy.deepcopy(cfg)
    exp_cfg['cnn'].update(overrides)

    random.seed(SEED); np.random.seed(SEED)
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

    m = build_cnn(exp_cfg, n_features)
    hist, ckpt = train_model(
        m, X_train, y_train, X_val, y_val,
        cfg=exp_cfg, model_key='cnn',
        checkpoint_dir='../results/checkpoints',
    )
    best_f1 = round(max(hist['val_f1']), 4)
    log_experiment({
        'exp_id': exp_id,
        'model': 'CNN',
        'num_filters': exp_cfg['cnn']['num_filters'],
        'kernel_size': exp_cfg['cnn']['kernel_size'],
        'dropout': exp_cfg['cnn']['dropout'],
        'best_val_f1': best_f1,
        'notes': notes,
    }, csv_path=str(DRIVE_ROOT / "metrics_summary.csv"))
    print(f'[{exp_id}] best_val_f1={best_f1}')
    return hist, ckpt, best_f1

In [ ]:
# Exp 02 — more filters
hist_02, _, f1_02 = run_cnn_experiment('cnn_02_filters128', {'num_filters': 128}, 'num_filters=128')

In [ ]:
# Exp 03 — fewer filters
hist_03, _, f1_03 = run_cnn_experiment('cnn_03_filters32', {'num_filters': 32}, 'num_filters=32')

In [ ]:
# Exp 04 — larger kernel
hist_04, _, f1_04 = run_cnn_experiment('cnn_04_kernel5', {'kernel_size': 5}, 'kernel_size=5')

In [ ]:
# Exp 05 — higher dropout
hist_05, _, f1_05 = run_cnn_experiment('cnn_05_dropout05', {'dropout': 0.5}, 'dropout=0.5')

In [ ]:
# Exp 06 — lower learning rate
hist_06, _, f1_06 = run_cnn_experiment('cnn_06_lr0001', {'learning_rate': 0.0001}, 'lr=0.0001')

## Final Evaluation on Test Set

In [ ]:
import pandas as pd
from src.evaluate import evaluate_dl_model
from src.utils import plot_loss_curves

results = pd.read_csv(str(DRIVE_ROOT / "metrics_summary.csv"))
cnn_results = results[results['model'] == 'CNN'].copy()
avail = [c for c in ['exp_id', 'num_filters', 'kernel_size', 'dropout', 'best_val_f1', 'notes'] if c in cnn_results.columns]
print(cnn_results[avail])

_hist_map = {
    'cnn_01_baseline':  history_baseline,
    'cnn_02_filters128': hist_02,
    'cnn_03_filters32':  hist_03,
    'cnn_04_kernel5':    hist_04,
    'cnn_05_dropout05':  hist_05,
    'cnn_06_lr0001':     hist_06,
}
best_row = cnn_results.loc[cnn_results['best_val_f1'].idxmax()]
print(f'\nBest experiment: {best_row["exp_id"]}  val_F1={best_row["best_val_f1"]}')
best_history = _hist_map.get(best_row['exp_id'], history_baseline)

In [ ]:
best_model = build_cnn(cfg, n_features).to(device)
best_model.load_state_dict(torch.load('../results/checkpoints/best_cnn.pt', map_location=device))

y_pred_cnn, y_prob_cnn = evaluate_dl_model(
    best_model, X_test, y_test,
    model_name='CNN',
    history=best_history,
    figures_dir='../results/figures',
    csv_path=str(DRIVE_ROOT / "metrics_summary.csv"),
)

np.save('../results/cnn_y_prob.npy', y_prob_cnn)

In [ ]:
plot_loss_curves(
    best_history['train_loss'], best_history['val_loss'],
    title='CNN Loss Curves (best)',
    save_path='../results/figures/loss_cnn.png',
)

In [ ]:
import shutil
shutil.copy('../results/checkpoints/best_cnn.pt', str(DRIVE_ROOT / 'best_cnn.pt'))
np.save(str(DRIVE_ROOT / 'cnn_y_prob.npy'), y_prob_cnn)
print(f'Checkpoint + probs saved to Drive: {DRIVE_ROOT}')